In [13]:
import os
import h5py
import numpy as np
from astropy.io import fits
from astropy.table import Table
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [37]:
gal_dataset = Table.read('/pscratch/sd/q/qshimp/Sorter/sga2025_1000test_with_anchors.csv')
cutout_dir = ""
h5_filename = "2025_Test1.h5"

num_galaxies = len(gal_dataset)
count = num_galaxies
pixel = 152

max_workers = 4      # safer than 8 or 16
chunk_size = 1000    # only submit 1000 jobs at a time
batch_size = 64      # write 64 images at a time
string_dtype = h5py.string_dtype(encoding="utf-8")

In [39]:
# Functions
def get_path(i):
    return (
        f"{cutout_dir}/"
        f"{int(gal_dataset['targetid'][i])}_grz_152_"
        f"{gal_dataset['target_ra'][i]:.4f}_"
        f"{gal_dataset['target_dec'][i]:.4f}.fits"
    )


def read_one(i):
    #path = get_path(i)
    path = gal_dataset['Path'][i]

    if not os.path.exists(path):
        print(path)
        return None

    with fits.open(path, memmap=False, ignore_missing_simple=True, ignore_missing_end=True) as hdul:
        img_data = hdul[0].data

    if img_data is None or img_data.shape != (3, pixel, pixel):
        return None

    return {
        "image": img_data.astype(np.float32),
        "ra": float(gal_dataset["target_ra"][i]),
        "dec": float(gal_dataset["target_dec"][i]),
        "mag_g": float(gal_dataset["g_mag"][i]),
        "mag_r": float(gal_dataset["r_mag"][i]),
        "mag_z": float(gal_dataset["z_mag"][i]),
        "cutout_type": 1,
        "Main_Type": int(gal_dataset["Main_type"][i]),
        "targetid": int(gal_dataset["targetid"][i]),
    }

In [40]:
# Run generator
with h5py.File(h5_filename, "w") as f:

    images = f.create_dataset(
        "images",
        (count, 3, pixel, pixel),
        dtype="f4",
        chunks=(64, 3, pixel, pixel),
        maxshape=(None, 3, pixel, pixel)
    )

    ra = f.create_dataset("ra", (count,), dtype="f8", maxshape=(None,))
    dec = f.create_dataset("dec", (count,), dtype="f8", maxshape=(None,))
    mag_g = f.create_dataset("mag_g", (count,), dtype="f4", maxshape=(None,))
    mag_r = f.create_dataset("mag_r", (count,), dtype="f4", maxshape=(None,))
    mag_z = f.create_dataset("mag_z", (count,), dtype="f4", maxshape=(None,))

    cutout_type = f.create_dataset("cutout_type", (count,), dtype="i4", maxshape=(None,))
    Main_Type = f.create_dataset("Main_Type", (count,), dtype="i4", maxshape=(None,))
    targetid = f.create_dataset("targetid", (count,), dtype="i8", maxshape=(None,))

    valid_entries = 0
    batch = []

    with tqdm(total=num_galaxies, desc="Saving galaxies") as pbar:

        for start in range(0, len(gal_dataset), chunk_size):

            if valid_entries >= num_galaxies:
                break

            end = min(start + chunk_size, len(gal_dataset))

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(read_one, i) for i in range(start, end)]

                for future in as_completed(futures):
                    try:
                        result = future.result()
                    except Exception as e:
                        print(f"Read error: {e}")
                        continue

                    if result is None:
                        continue

                    batch.append(result)

                    if len(batch) >= batch_size:
                        n = len(batch)
                        s = slice(valid_entries, valid_entries + n)

                        images[s] = np.stack([b["image"] for b in batch])
                        ra[s] = [b["ra"] for b in batch]
                        dec[s] = [b["dec"] for b in batch]
                        mag_g[s] = [b["mag_g"] for b in batch]
                        mag_r[s] = [b["mag_r"] for b in batch]
                        mag_z[s] = [b["mag_z"] for b in batch]
                        cutout_type[s] = [b["cutout_type"] for b in batch]
                        Main_Type[s] = [b["Main_Type"] for b in batch]
                        targetid[s] = [b["targetid"] for b in batch]

                        valid_entries += n
                        pbar.update(n)
                        batch = []

            f.flush()

    # Write final leftover batch
    if len(batch) > 0:
        n = len(batch)
        s = slice(valid_entries, valid_entries + n)

        images[s] = np.stack([b["image"] for b in batch])
        ra[s] = [b["ra"] for b in batch]
        dec[s] = [b["dec"] for b in batch]
        mag_g[s] = [b["mag_g"] for b in batch]
        mag_r[s] = [b["mag_r"] for b in batch]
        mag_z[s] = [b["mag_z"] for b in batch]
        cutout_type[s] = [b["cutout_type"] for b in batch]
        Main_Type[s] = [b["Main_Type"] for b in batch]
        targetid[s] = [b["targetid"] for b in batch]

        valid_entries += n

    print(f"Saved {valid_entries} valid galaxies to {h5_filename}.")

Task was destroyed but it is pending!
task: <Task pending name='Task-213' coro=<_async_in_context.<locals>.run_in_context() done, defined at /global/common/software/desi/perlmutter/desiconda/20260227-2.3.1/conda/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-214' coro=<Kernel.shell_main() running at /global/common/software/desi/perlmutter/desiconda/20260227-2.3.1/conda/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /global/common/software/desi/perlmutter/desiconda/20260227-2.3.1/conda/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/global/common/software/desi/perlmutter/desiconda/20260227-2.3.1/conda/lib/python3.13/site-packages/traitlets/traitlets.py:222: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def __init__(self, value: t.Any) -> None:
Task was destroyed but it is pending!
task: <Task pending name='Task-214' coro=<Kern

Saving galaxies:   0%|          | 0/5000 [00:00<?, ?it/s]

/tmp/ipykernel_605780/1830454114.py:29: UserWarning: Warning: converting a masked element to nan.
  "mag_g": float(gal_dataset["g_mag"][i]),
/tmp/ipykernel_605780/1830454114.py:31: UserWarning: Warning: converting a masked element to nan.
  "mag_z": float(gal_dataset["z_mag"][i]),


Saved 5000 valid galaxies to 2025_Test1.h5.
